In [10]:
import yfinance as yf
import backtrader as bt
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt

In [18]:
runBacktest("NVDA", SmaCross, True)

[*********************100%***********************]  1 of 1 completed


Starting Portfolio Value: 10000.00
2025-02-28, LONG ORDER CREATE, 125.05
Order completed: 123.50499725341797
2025-03-03, STOP LOSS HIT - CLOSING POSITION, 120.19
Order completed: 120.1500015258789
2025-03-03, SHORT ORDER CREATE, 119.54
Order completed: 119.5199966430664
2025-03-03, STOP LOSS HIT - CLOSING POSITION, 118.62
Order completed: 118.61000061035156
2025-03-04, LONG ORDER CREATE, 116.20
Order completed: 116.20999908447266
2025-03-04, STOP LOSS HIT - CLOSING POSITION, 116.78
Order completed: 116.77999877929688
2025-03-05, SHORT ORDER CREATE, 114.65
Order completed: 114.75
2025-03-05, STOP LOSS HIT - CLOSING POSITION, 115.89
Order completed: 115.87000274658203
2025-03-05, LONG ORDER CREATE, 117.96
Order completed: 117.9800033569336
2025-03-05, STOP LOSS HIT - CLOSING POSITION, 117.11
Order completed: 117.11000061035156
2025-03-06, SHORT ORDER CREATE, 114.84
Order completed: 114.83999633789062
2025-03-06, STOP LOSS HIT - CLOSING POSITION, 115.14
Order completed: 115.12000274658203

In [14]:
class SmaCross(bt.Strategy):
    def log(self, txt, dt=None):
        """Logging function fot this strategy"""
        dt = dt or self.datas[0].datetime.date(0)
        print("%s, %s" % (dt.isoformat(), txt))

    def __init__(self):

        self.bars = 0

        self.dataclose = self.datas[0]
        self.sma1 = bt.ind.SMA(period=10)
        self.sma2 = bt.ind.SMA(period=30)

        self.long_tp = 0.05
        self.long_drawdown = 0.01

        self.short_tp = 0.05
        self.short_drawdown = 0.01

        self.long_tp_price = float("inf")
        self.long_drawdown_price = 0

        self.short_tp_price = 0
        self.short_drawdown_price = float("inf")

        self.short_max = float("inf")
        self.long_max = 0

    def next(self):
        self.bars += 1
        if self.datas[0].datetime.date(0) == datetime.today():
            print("CLSOSLDFHPOSDFIHS")
            if self.position:
                self.close()
                self.env.runstop()
            
            

        if len(self) == len(self.data) - 1 and self.position:
            print("Closing on the last bar")
            self.close()

        if not self.position:  # not in the market
            if self.sma1[0] > self.sma2[0] and self.sma1[-1] < self.sma2[-1]:
                self.log("LONG ORDER CREATE, %.2f" % self.dataclose[0])

                self.order = self.buy(size=10)
                self.long_tp_price = self.dataclose[0] * (1 + self.long_tp)

                self.long_max = self.dataclose[0]

                self.notify_order(self.order)

            elif self.sma1[0] < self.sma2[0] and self.sma1[-1] > self.sma2[-1]:
                self.log("SHORT ORDER CREATE, %.2f" % self.dataclose[0])

                self.order = self.sell(size=10)
                self.short_tp_price = self.dataclose[0] * (1 - self.short_tp)
                self.short_max = self.dataclose[0]

                self.notify_order(self.order)
        else:
            self.short_max = min(self.dataclose[0], self.short_max)
            self.long_max = max(self.dataclose[0], self.long_max)

            self.short_drawdown_price = self.short_max * (1 - self.short_drawdown)

            self.long_drawdown_price = self.long_max * (1 - self.long_drawdown)

            # Short Stop loss
            if self.dataclose[0] >= self.short_drawdown_price:
                self.log("STOP LOSS HIT - CLOSING POSITION, %.2f" % self.dataclose[0])
                self.close()

            # Short Take profit
            elif self.dataclose[0] <= self.short_tp_price:
                self.log("TAKE PROFIT HIT - CLOSING POSITION, %.2f" % self.dataclose[0])
                self.close()

            # Long Stop loss
            elif self.dataclose[0] <= self.long_drawdown_price:
                self.log("STOP LOSS HIT - CLOSING POSITION, %.2f" % self.dataclose[0])
                self.close()

            # Long Take profit
            elif self.dataclose[0] >= self.long_tp_price:
                self.log("TAKE PROFIT HIT - CLOSING POSITION, %.2f" % self.dataclose[0])
                self.close()

    def notify_order(self, order):
        if order.status in [order.Submitted, order.Accepted]:
            return
        if order.status == order.Completed:
            print(f"Order completed: {order.executed.price}")
        elif order.status == order.Canceled:
            print("Order canceled")
        elif order.status == order.Rejected:
            print("Order rejected")

    def stop(self):
        if self.position:
            print("Closing position at end of backtest")
            self.close()


In [16]:
def runBacktest(ticker, strategy, plot = False):

    filepath = f"/home/jacob/Documents/Shoptaki/futures2/Future-Trading-Platform/backend/backtesting/{ticker.lower()}_data.csv"

    df = pd.DataFrame(
        yf.download(ticker, start="2025-02-27", end=datetime.today(), interval="15m")
    )

    df.columns = ["Close", "High", "Low", "Open", "Volume"]

    df.to_csv(filepath)


    data = bt.feeds.GenericCSVData(
        dataname=filepath,
        dtformat="%Y-%m-%d %H:%M:%S%z",
        timeframe=bt.TimeFrame.Minutes,
        compression=15,
        datetime=0,
        open=4,
        high=2,
        low=3,
        close=1,
        volume=5,
        openinterest=-1,
    )

    cerebro = bt.Cerebro()
    cerebro.addstrategy(strategy)
    cerebro.adddata(data)

    cerebro.broker.set_cash(10000)
    cerebro.broker.setcommission(commission=0.001)  # 0.1%

    print("Starting Portfolio Value: %.2f" % cerebro.broker.getvalue())

    cerebro.run(runonce=True)

    print("Final Portfolio Value: %.2f" % cerebro.broker.getvalue())

    plt.rcParams.update(
    {
        "font.size": 14,  # Increase this for bigger text
        "axes.titlesize": 16,
        "axes.labelsize": 14,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "legend.fontsize": 12,
    }
    )

    cerebro.plot(iplot=False, figsize=(18, 10), dpi=480)  # Width x Height in inches
